# nlp text human emotions project

## dependencies


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)
from xgboost import XGBClassifier

import warnings
warnings.filterwarnings("ignore")

In [3]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

## dataset

In [4]:
# kaggle api  configuration
!mkdir ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

mkdir: cannot create directory ‘/root/.kaggle’: File exists


In [5]:
# downloading the dataset
!kaggle datasets download -d prajwalnayakat/text-emotion

Dataset URL: https://www.kaggle.com/datasets/prajwalnayakat/text-emotion
License(s): Attribution 4.0 International (CC BY 4.0)
text-emotion.zip: Skipping, found more recently modified local copy (use --force to force download)


In [6]:
!ls

final_dataset.csv  kaggle.json	sample_data  text-emotion.zip


In [ ]:
# unzipping the dataset
!unzip /content/text-emotion.zip -d /content/

Archive:  /content/text-emotion.zip
replace /content/final_dataset.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
!ls

In [ ]:
# dataset
data = pd.read_csv("/content/final_dataset.csv")
data.head()

## data exploration and cleaning

In [ ]:
data.shape

In [ ]:
data.columns

In [ ]:
len(data["emotion"].value_counts())

In [ ]:
data["emotion"].value_counts()

In [ ]:
# sample per emotions
def per_emootions_sample(data, samples=5):
  emotions = data["emotion"].unique().tolist()
  for emotion in emotions:
      print(f"--- samples for {emotion} ---")
      print(data[data["emotion"] == emotion].sample(samples))
      print("\n")

In [ ]:
per_emootions_sample(data=data)

In [ ]:
# checking for null values
data.isnull().sum()

In [ ]:
data.shape

## explortory data analysis

In [ ]:
# emotion distribution
order = data["emotion"].value_counts().index

plt.figure(figsize=(8, 5))
sns.countplot(
    data = data,
    x = "emotion",
    order = order,
    hue = "emotion",
    palette = "magma",
    legend = False
)
plt.title("emotion distribution")
plt.xlabel("emotion")
plt.ylabel("count")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# text lenmgth analysis (words)
data["text_length"] = data["text"].apply(
    lambda x: len(str(x).split())
)

In [ ]:
data.head()

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(
    data = data,
    x = "emotion",
    y = "text_length",
    hue = "emotion",
    palette = "pastel"
)
plt.xlabel("emotion")
plt.ylabel("number of words")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# statistical summary of length
data["text_length"].describe()

## texts preprocessing...........

In [ ]:
#  initializing nlp tools
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [ ]:
# regex text cleaning function
def text_cleaner(text):
  text = text.lower()
  text = re.sub(r"http\S+|www\S+", "", text)
  text = re.sub(r"\d+", "", text)
  text = text.translate(str.maketrans('', '', string.punctuation))
  tokens = text.split()
  tokens = [word for word in tokens if word not in stop_words]
  tokens = [lemmatizer.lemmatize(word) for word in tokens]
  return " ".join(tokens)

In [ ]:
# applying text cleaner
data["clean_text"] = data["text"].apply(text_cleaner)
display(data[['text', 'clean_text']].sample(5))

## vectorization

In [ ]:
# initializing tf-idf vectorizer
vectorizer = TfidfVectorizer(
    max_features = 5000,
    ngram_range = (1, 2)
)

In [ ]:
# fir adn transform clean text
X = vectorizer.fit_transform(data["clean_text"])

In [ ]:
# encoding target labels
encoder = LabelEncoder()
y = encoder.fit_transform(data["emotion"])

In [ ]:
encoder.classes_

In [ ]:
print(
    X.shape,
    y.shape
)

In [ ]:
# training and testing splits
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size = 0.2,
    random_state = 21,
    stratify = y
)

print("X shape:", X.shape)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

## modelling, training and evaluation

In [ ]:
# base models
models = {
    "multinomial naive bayes": MultinomialNB(),
    "logistic regression": LogisticRegression(max_iter=1000),
    "linear svc": LinearSVC()
}

In [ ]:
for name, model in models.items():
  print(f"{name}: {model}")

In [ ]:
# models result
model_results = {}

In [ ]:
X_train

In [ ]:
# training, prediction and evaluation of the base models
for model_name, model in models.items():

  model.fit(X_train, y_train)

  y_pred = model.predict(X_test)
  acc = accuracy_score(y_test, y_pred)
  report = classification_report(
      y_test, y_pred, target_names = encoder.classes_
  )

  #results
  model_results[model_name] = {
      "accuracy": acc,
      "classification report": report
  }

  print(f"====== {model_name} ======")
  print("accuracy:", acc)
  print("classification report:\n", report)
  print("\n")
